# 3. Custom Tools for a Customer Outreach Campaign

**Learning objectives**
- Understand why agents need tools.
- Use a web-search tool when a Serper API key is available.
- Create a custom CrewAI tool with a validated input schema.
- Use a deterministic sentiment-analysis demonstration.
- Generate email drafts while keeping a human in control of sending.

# Setup: CrewAI + Google Gemini (Colab)

**Before running:** add a Gemini API key to Google Colab → Secrets with the name `GOOGLE_API_KEY` and enable notebook access to it. A Serper key is optional and is only needed for live web search in the relevant exercises.

These notebooks use the current CrewAI Python API style. The exact package versions are intentionally not pinned to an old 0.x release; after a successful lab setup, freeze the working environment for repeatability.

In [ ]:
%pip -q install -U "crewai[google-genai]" crewai-tools

In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("Add GOOGLE_API_KEY in Colab Secrets and enable notebook access.")

llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
    temperature=0.2
)
print("LLM configured.")

## Step 1 — Configure an optional web-search tool

Add `SERPER_API_KEY` to Colab Secrets to enable live search. If it is not configured, this notebook still runs using the local demo tool and supplied fictional company brief.

In [ ]:
from crewai_tools import SerperDevTool

SERPER_API_KEY = userdata.get("SERPER_API_KEY")
search_tool = None

if SERPER_API_KEY:
    os.environ["SERPER_API_KEY"] = SERPER_API_KEY
    search_tool = SerperDevTool()
    print("Live web search enabled.")
else:
    print("SERPER_API_KEY not found: live search disabled.")

## Step 2 — Build a custom sentiment tool

This is a transparent **rule-based teaching example**, not a trained sentiment model. It counts simple positive and negative words. Discuss its limitations: negation, sarcasm, context and domain-specific language.

In [ ]:
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

class SentimentInput(BaseModel):
    text: str = Field(..., description="Text whose sentiment should be estimated.")

class SimpleSentimentTool(BaseTool):
    name: str = "Simple Rule-Based Sentiment"
    description: str = (
        "Estimate text sentiment using a small positive/negative word list. "
        "This is a simple demonstration, not a production sentiment model."
    )
    args_schema: type[BaseModel] = SentimentInput

    def _run(self, text: str) -> str:
        positive_words = {
            "good", "great", "excellent", "helpful", "success",
            "interested", "pleased", "improved", "benefit", "love"
        }
        negative_words = {
            "bad", "poor", "issue", "problem", "delay",
            "disappointed", "difficult", "failed", "hate", "complaint"
        }

        tokens = set(re.findall(r"[a-z']+", text.lower()))
        positive = len(tokens & positive_words)
        negative = len(tokens & negative_words)

        if positive > negative:
            label = "positive"
        elif negative > positive:
            label = "negative"
        else:
            label = "neutral"

        return (
            f"Estimated sentiment: {label}. "
            f"Positive keyword matches: {positive}; negative keyword matches: {negative}. "
            "This is a rule-based estimate and may miss context or sarcasm."
        )

sentiment_tool = SimpleSentimentTool()
print(sentiment_tool.run(text="The support team was helpful and the launch was great."))

## Step 3 — Define the agents

The first agent profiles the organisation. The second drafts a personalized message based on the profile. Do not assume that a named individual is the decision-maker unless the source confirms it.

In [ ]:
tools_for_research = [search_tool] if search_tool else []

lead_researcher = Agent(
    role="Organisation Research Analyst",
    goal="Prepare a concise, evidence-aware profile of {lead_name}",
    backstory=(
        "You research organisations using available tools. Separate verified "
        "facts from assumptions, include source links when web search is used, "
        "and report when information cannot be verified."
    ),
    tools=tools_for_research,
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

outreach_writer = Agent(
    role="Customer Outreach Writer",
    goal="Create a relevant, respectful and personalized outreach email draft",
    backstory=(
        "You write professional outreach. Use only facts present in the "
        "research brief or supplied inputs. Do not invent personal details, "
        "relationships, product claims or recent milestones."
    ),
    tools=[sentiment_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

## Step 4 — Define the tasks

In [ ]:
profile_task = Task(
    description=(
        "Prepare a profile of {lead_name}, operating in {industry}. "
        "Use web search if available. If live search is unavailable, rely only "
        "on the supplied brief and clearly state that independent verification "
        "has not been performed. Do not infer who the decision-maker is."
    ),
    expected_output=(
        "A short profile with verified facts, source URLs if available, "
        "possible needs and a separate list of unknowns."
    ),
    agent=lead_researcher
)

outreach_task = Task(
    description=(
        "Using the profile of {lead_name}, draft an outreach email to "
        "{contact_name}, whose supplied role is {contact_role}. "
        "The supplied context is: {context}. "
        "Do not add unsupported facts. Use the sentiment tool to review the "
        "tone of the draft. Return a subject line and email body only."
    ),
    expected_output="One concise, personalized email draft with a subject line and body.",
    agent=outreach_writer,
    tools=[sentiment_tool],
    markdown=True
)

## Step 5 — Assemble and run the Crew

`human_input=True` asks for human review at the task stage in supported execution environments. In this exercise, treat the result as a draft: do not connect an email-sending tool.

In [ ]:
outreach_crew = Crew(
    agents=[lead_researcher, outreach_writer],
    tasks=[profile_task, outreach_task],
    process=Process.sequential,
    verbose=True
)

# Fictional example data for classroom use.
inputs = {
    "lead_name": "Northstar Learning Pvt Ltd",
    "industry": "Online education",
    "contact_name": "Aarav Mehta",
    "contact_role": "Partnerships Manager",
    "context": (
        "The organisation is exploring ways to improve learner engagement. "
        "This is a fictional scenario supplied for the exercise, not a verified milestone."
    )
}

outreach_result = outreach_crew.kickoff(inputs=inputs)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(outreach_result.raw))

## Step 6 — Human review before use

The crew produces a draft only. Review the recipient, factual claims, tone and relevance manually. This simple cell records a classroom approval decision; it does not send an email or call an external service.

In [ ]:
approval = input("After reviewing the draft, type APPROVE to mark it ready for manual use: ").strip()
if approval == "APPROVE":
    print("Marked as reviewed. No email has been sent.")
else:
    print("Not approved. Revise the draft before use.")

## Try it yourself
1. Test the sentiment tool with positive, negative, mixed and sarcastic sentences.
2. Add a custom tool that checks whether an email contains a greeting, clear purpose and closing.
3. Run with and without Serper search and compare the evidence quality.
4. Add a human approval step before any external communication.

**Safety rule:** Keep the workflow in draft mode. Never send bulk or personalized messages automatically from a classroom exercise.